# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [42]:
# My code
df_raw = pd.read_csv("AviationData.csv", encoding="latin1", low_memory=False)
print("Shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(15))
display(df_raw.describe(include="all").T.head(20))

Shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

None

,missing_count
Schedule,76307
Air.carrier,72241
FAR.Description,56866
Aircraft.Category,56602
Longitude,54516
Latitude,54507
Airport.Code,38757
Airport.Name,36185
Broad.phase.of.flight,27165
Publication.Date,13771


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [43]:
# Making sure to keep records from 1983 onwards
df = df_raw.copy()
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df = df.loc[df["Event.Date"].dt.year >= 1983].copy()

# Focusing on the accidents that involved fixed wing airplanes but the source contains many missing aircraft records
df["Aircraft.Category"] = df["Aircraft.Category"].astype("string").str.strip()
df["Aircraft.Category"] = df["Aircraft.Category"].replace({
    "WSFT": "Weight-Shift", "ULTR": "Ultralight", "UNK": pd.NA, "Unknown": pd.NA
})
df = df.loc[
    df["Investigation.Type"].astype("string").str.strip().str.casefold().eq("accident")
].copy()

# Whitespace and missing common values in the text columns
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({
        "": pd.NA, "Unknown": pd.NA, "UNKNOWN": pd.NA,
        "UNK": pd.NA, "Unk": pd.NA, "N/A": pd.NA, "NA": pd.NA
    })

print("Rows after date & accident filtering:", len(df))
print("Date range:", df["Event.Date"].min(), "to", df["Event.Date"].max())
display(df[["Event.Date","Aircraft.Category","Make","Model","Investigation.Type"]].head())

Rows after date & accident filtering: 81534
Date range: 1983-01-01 00:00:00 to 2022-12-29 00:00:00


,Event.Date,Aircraft.Category,Make,Model,Investigation.Type
3600,1983-01-01,<NA>,Piccard,AX-6,Accident
3601,1983-01-01,<NA>,Cessna,182P,Accident
3602,1983-01-01,<NA>,Cessna,182RG,Accident
3603,1983-01-01,<NA>,Cessna,182P,Accident
3604,1983-01-01,<NA>,Piper,PA-28R-200,Accident


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [44]:
# clean injury columns so they are numbers, filling blanks with 0
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# calculate total people on board and serious/fatal injuries
df['total_occupants'] = df[injury_cols].sum(axis=1)
df['serious_fatal_count'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']

# calculate injury fractions (avoiding division by zero)
df['injury_fraction'] = df['serious_fatal_count'] / df['total_occupants']
df['injury_fraction'] = df['injury_fraction'].replace([float('inf'), -float('inf')], np.nan)

# check summary stats
display(df[['total_occupants', 'serious_fatal_count', 'injury_fraction']].describe())

,total_occupants,serious_fatal_count,injury_fraction
count,81534.000000,81534.000000,81083.000000
mean,4.215836,0.845316,0.301604
std,20.652704,5.657502,0.442025
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000
max,699.000000,349.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [45]:
# Cleaning aircraft damage column and creating a binary destroyed column (1 if destroyed, 0 if not)
df['Aircraft.damage'] = df['Aircraft.damage'].astype(str).str.strip().str.title()
df['Aircraft.damage'] = df['Aircraft.damage'].replace(['Unknown', 'Unk', 'nan'], np.nan)

df['aircraft_destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
df.loc[df['Aircraft.damage'].isna(), 'aircraft_destroyed'] = np.nan

print("Aircraft damage value counts:")
display(df['Aircraft.damage'].value_counts(dropna=False))

print("Aircraft destroyed indicator counts:")
display(df['aircraft_destroyed'].value_counts(dropna=False))

Aircraft damage value counts:


Aircraft.damage
Substantial    61733
Destroyed      17567
NaN             1525
Minor            709
Name: count, dtype: int64

Aircraft destroyed indicator counts:


aircraft_destroyed
0.0    62442
1.0    17567
NaN     1525
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [46]:
# Cleaning and standardizing the Make column
df['Make'] = df['Make'].astype(str).str.strip().str.title()

# Handling/fixing common manufacturer name variations
make_aliases = {
    'Mcdonnell Douglas': 'McDonnell Douglas',
    'Mcdonnell-Douglas': 'McDonnell Douglas',
    'Airbus Industrie': 'Airbus',
    'Boeing Company': 'Boeing',
    'Cessna Aircraft Company': 'Cessna',
    'Piper Aircraft Inc': 'Piper',
    'Dehavilland': 'De Havilland'
}
df['Make'] = df['Make'].replace(make_aliases)

# Keeping only the makes with at least 50 records
make_counts = df['Make'].value_counts()
popular_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(popular_makes)].copy()

print(f"Rows remaining: {len(df)}")
print(f"Number of makes kept: {df['Make'].nunique()}")
display(df['Make'].value_counts().head(20).to_frame('accident_records'))

Rows remaining: 67712
Number of makes kept: 88


,accident_records
Make,
Cessna,25490
Piper,14005
Beech,4929
Bell,2553
Boeing,1329
Mooney,1256
Robinson,1196
Grumman,1055
Bellanca,978


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [47]:
# Dropping rows with missing make or model, and formatting model names
df['Model'] = df['Model'].astype(str).str.strip().str.upper()
df = df.dropna(subset=['Model', 'Make']).copy()

# Creating a combined Make and Model identifier
df['Make.Model'] = df['Make'] + " | " + df['Model']

# Standardizing text in key categorical columns
cat_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()
        df[col] = df[col].replace({'Vmc': 'VMC', 'Imc': 'IMC', 'Vfr': 'VFR', 'Ifr': 'IFR'})

# Checking the top makes with the most unique models
print("Top makes by unique model count:")
display(df.groupby('Make')['Model'].nunique().sort_values(ascending=False).head(10).to_frame('unique_models'))

print("Sample of combined Make.Model identifiers:")
display(df[['Make', 'Model', 'Make.Model']].head())

Top makes by unique model count:


,unique_models
Make,
Cessna,805
Piper,630
Beech,524
Boeing,354
Bell,285
Schweizer,138
Aerospatiale,121
McDonnell Douglas,113
De Havilland,112


Sample of combined Make.Model identifiers:


,Make,Model,Make.Model
3601,Cessna,182P,Cessna | 182P
3602,Cessna,182RG,Cessna | 182RG
3603,Cessna,182P,Cessna | 182P
3604,Piper,PA-28R-200,Piper | PA-28R-200
3605,Cessna,140,Cessna | 140


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [48]:
# Cleaning and standardizing analysis columns, keeping missing values as NaN
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

cat_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace(['Unknown', 'Unk', 'Nan', 'None'], np.nan)

# Specific weather abbreviations formatting
df['Weather.Condition'] = df['Weather.Condition'].replace({'Vmc': 'VMC', 'Imc': 'IMC'})

# Checking missing values across these fields
print("Missing values in key analysis fields:")
display(df[cat_cols + ['Number.of.Engines']].isna().sum().to_frame('missing_count'))

Missing values in key analysis fields:


,missing_count
Engine.Type,5891
Weather.Condition,3407
Purpose.of.flight,8056
Broad.phase.of.flight,19414
Number.of.Engines,3894


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [49]:
# Dropping columns that have too many NaNs for our analysis
drop_cols = [
    'Event.Id', 'Accident.Number', 'Location', 'Latitude', 'Longitude',
    'Airport.Code', 'Airport.Name', 'Registration.Number', 'FAR.Description',
    'Schedule', 'Air.carrier', 'Report.Status', 'Publication.Date',
    'Amateur.Built', 'Country', 'Investigation.Type', 'Injury.Severity'
]

df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

print(f"Cleaned dataset shape: {df_clean.shape}")
print("Fraction of missing values per column:")
display(df_clean.isna().mean().sort_values(ascending=False).to_frame('missing_fraction'))

Cleaned dataset shape: (67694, 19)
Fraction of missing values per column:


,missing_fraction
Aircraft.Category,0.699501
Broad.phase.of.flight,0.286791
Purpose.of.flight,0.119006
Engine.Type,0.087024
Number.of.Engines,0.057524
Weather.Condition,0.050329
Aircraft.damage,0.018332
aircraft_destroyed,0.018332
injury_fraction,0.005554
Event.Date,0.000000


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [50]:
# Saving and printing the cleaned dataframe to a new csv file
output_path = "AviationData_Cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"Successfully saved {output_path} with {len(df_clean):,} rows and {df_clean.shape[1]} columns.")

Successfully saved AviationData_Cleaned.csv with 67,694 rows and 19 columns.
